# AutoQSAR Benchmark Summary

This notebook reads the output from `run_autoqsar_ga_benchmarks.py` and summarizes model performance across datasets.

By default it loads the most recent run under `./benchmark_results/`, then builds:
- a model-performance summary across datasets
- a leaderboard-relative summary using cached top-10 reference artifacts from the run directory
- diagnostic visuals for model spread, predictions, and (if present) GA convergence
- feature-family selection representation analysis from selector outputs (coverage, enrichment, and drop/watchlist suggestions)



In [89]:
# @title 0. Choose a benchmark run { display-mode: "form" }
benchmark_run_dir = "AUTO"  # @param {type:"string"}
show_top_n_models = 10  # @param {type:"slider", min:3, max:20, step:1}
rmse_metric_column_preference = "AUTO"  # @param {type:"string"}
prediction_sample_cap = 5000  # @param {type:"integer"}


In [90]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def display_note(text):
    display(Markdown(text))


def resolve_benchmark_dir(path_text="AUTO"):
    path_text = str(path_text).strip()
    if path_text and path_text.upper() != "AUTO":
        candidate = Path(path_text).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"Benchmark directory not found: {candidate}")
        return candidate

    root = Path("./benchmark_results").resolve()
    if not root.exists():
        raise FileNotFoundError("No ./benchmark_results directory exists yet.")
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No benchmark run directories were found under ./benchmark_results.")
    return candidates[0]


def read_optional_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


def read_optional_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def pick_column(columns, candidates):
    cols = list(columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    lowered = {str(col).strip().lower(): col for col in cols}
    for candidate in candidates:
        match = lowered.get(str(candidate).strip().lower())
        if match is not None:
            return match
    return None


def metric_column(metric_df, preferred="AUTO", metric_hint="rmse"):
    if metric_df is None or metric_df.empty:
        return None
    if str(preferred).strip() and str(preferred).strip().upper() != "AUTO":
        if preferred not in metric_df.columns:
            raise ValueError(f"Requested metric column not found: {preferred}")
        return preferred
    candidates = []
    hint = str(metric_hint).strip().lower()
    if hint == "rmse":
        candidates = [
            "test_rmse", "Test RMSE", "cv_rmse", "CV RMSE", "mean_test_rmse",
            "rmse", "RMSE"
        ]
    elif hint == "r2":
        candidates = ["test_r2", "Test R2", "cv_r2", "CV R2", "r2", "R2"]
    return pick_column(metric_df.columns, candidates)


def standardize_metric_frame(metric_df):
    if metric_df is None:
        return None
    frame = metric_df.copy()
    rename_map = {}
    dataset_col = pick_column(frame.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(frame.columns, ["model", "Model"])
    workflow_col = pick_column(frame.columns, ["workflow", "Workflow"])
    if dataset_col is not None:
        rename_map[dataset_col] = "dataset"
    if model_col is not None:
        rename_map[model_col] = "model"
    if workflow_col is not None:
        rename_map[workflow_col] = "workflow"
    frame = frame.rename(columns=rename_map)
    return frame


In [91]:
RUN_DIR = resolve_benchmark_dir(benchmark_run_dir)
SUMMARY_METRICS = standardize_metric_frame(read_optional_csv(RUN_DIR / "summary_metrics.csv"))
TEST_RMSE_PIVOT = read_optional_csv(RUN_DIR / "test_rmse_pivot.csv")
PREDICTIONS = read_optional_csv(RUN_DIR / "predictions.csv")
GA_HISTORY = read_optional_csv(RUN_DIR / "ga_history.csv")
RUN_CONFIG = read_optional_json(RUN_DIR / "run_config.json")
LEADERBOARD_TOP10 = read_optional_csv(RUN_DIR / "leaderboard_top10_reference.csv")
LEADERBOARD_COMPARISON = read_optional_csv(RUN_DIR / "leaderboard_comparison_by_dataset.csv")

run_status_rows = []
for status_path in sorted(RUN_DIR.glob("*/run_status.json")):
    payload = read_optional_json(status_path)
    if not isinstance(payload, dict):
        continue
    run_status_rows.append(
        {
            "dataset": status_path.parent.name,
            "status": payload.get("status", "unknown"),
            "checkpoint_stage": payload.get("checkpoint_stage", ""),
            "n_metrics_rows": payload.get("n_metrics_rows", np.nan),
            "started_at": payload.get("started_at", ""),
            "completed_at": payload.get("completed_at", ""),
            "elapsed_seconds": payload.get("elapsed_seconds", np.nan),
        }
    )
RUN_STATUS = pd.DataFrame(run_status_rows)

dataset_metric_tables = []
for metrics_path in sorted(RUN_DIR.glob("*/metrics.csv")):
    dataset_name = metrics_path.parent.name
    frame = pd.read_csv(metrics_path)
    frame = standardize_metric_frame(frame)
    if frame is not None and not frame.empty and "dataset" not in frame.columns:
        frame["dataset"] = dataset_name
    dataset_metric_tables.append(frame)

if SUMMARY_METRICS is None and dataset_metric_tables:
    SUMMARY_METRICS = pd.concat(dataset_metric_tables, ignore_index=True)

if SUMMARY_METRICS is None or SUMMARY_METRICS.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

if "dataset" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

RMSE_COL = metric_column(SUMMARY_METRICS, rmse_metric_column_preference, metric_hint="rmse")
R2_COL = metric_column(SUMMARY_METRICS, "AUTO", metric_hint="r2")

print(f"Benchmark run directory: {RUN_DIR}")
print(f"Datasets: {SUMMARY_METRICS['dataset'].nunique()}")
print(f"Models: {SUMMARY_METRICS['model'].nunique()}")
print(f"Rows in summary table: {len(SUMMARY_METRICS):,}")
print(f"RMSE column used: {RMSE_COL}")
if R2_COL is not None:
    print(f"R2 column used: {R2_COL}")
if RUN_CONFIG is not None:
    display_note("Loaded `run_config.json` for this benchmark run.")

if not RUN_STATUS.empty:
    completed = int((RUN_STATUS["status"].astype(str).str.lower() == "completed").sum())
    print(f"Datasets with run_status.json: {len(RUN_STATUS)} (completed: {completed})")

if LEADERBOARD_TOP10 is not None:
    print(f"Leaderboard top-10 reference rows: {len(LEADERBOARD_TOP10):,}")
if LEADERBOARD_COMPARISON is not None:
    print(f"Leaderboard comparison rows: {len(LEADERBOARD_COMPARISON):,}")


Benchmark run directory: C:\Users\Scott.Coffin\OneDrive - California OEHHA\R_new\AutoQSAR\benchmark_results\all_benchmarks_no_unimol_20260419_223505
Datasets: 13
Models: 14
Rows in summary table: 176
RMSE column used: test_rmse
R2 column used: test_r2


Loaded `run_config.json` for this benchmark run.

Datasets with run_status.json: 14 (completed: 13)
Leaderboard top-10 reference rows: 144


In [92]:
config_summary = pd.DataFrame([RUN_CONFIG]) if isinstance(RUN_CONFIG, dict) else pd.DataFrame()
if not config_summary.empty:
    display(config_summary.T.rename(columns={0: "value"}))

best_by_dataset = SUMMARY_METRICS.sort_values(RMSE_COL, ascending=True).groupby("dataset", as_index=False).first()
best_by_dataset = best_by_dataset[[col for col in ["dataset", "workflow", "model", RMSE_COL, R2_COL] if col in best_by_dataset.columns]]
display_note("### Best model per dataset")
display(best_by_dataset.sort_values(RMSE_COL, ascending=True).reset_index(drop=True))

model_summary = (
    SUMMARY_METRICS.groupby([col for col in ["workflow", "model"] if col in SUMMARY_METRICS.columns], dropna=False)
    .agg(
        datasets=("dataset", "nunique"),
        mean_rmse=(RMSE_COL, "mean"),
        median_rmse=(RMSE_COL, "median"),
        std_rmse=(RMSE_COL, "std"),
        best_rmse=(RMSE_COL, "min"),
        worst_rmse=(RMSE_COL, "max"),
        mean_r2=(R2_COL, "mean") if R2_COL is not None else (RMSE_COL, "size"),
    )
    .reset_index()
)
if R2_COL is None and "mean_r2" in model_summary.columns:
    model_summary = model_summary.drop(columns=["mean_r2"])
display_note("### Cross-dataset model summary")
display(model_summary.sort_values("mean_rmse", ascending=True).head(int(show_top_n_models)).reset_index(drop=True))


,value
dataset,None
refresh_leaderboards_only,False
dataset_name,None
include_local_csv,None
output_dir,benchmark_results\all_benchmarks_no_unimol_202...
...,...
ensemble_exclude_negative_test_r2_members,True
resume,True
default_feature_families,"[morgan, ecfp6, fcfp6, layered, atom_pair, top..."
config_signature,"{""chemml_batch_size"": 64, ""chemml_cv_folds"": 5..."


### Best model per dataset

,dataset,workflow,model,test_rmse,test_r2
0,chemml_organic_density,ChemML deep learning,ChemML MLP (PyTorch),0.005161,0.972716
1,chemml_cep_homo,conventional,MapLight CatBoost,0.107977,0.976514
2,tdc_caco2_wang,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",0.352770,0.730860
3,esol_delaney,conventional,MapLight CatBoost,0.618938,0.817447
4,tdc_lipophilicity_astrazeneca,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",0.627152,0.720792
5,tdc_ld50_zhu,ChemML deep learning,ChemML MLP (TensorFlow),0.839571,0.370982
6,tdc_solubility_aqsoldb,conventional,XGBoost,1.023426,0.800833
7,freesolv_sampl,Chemprop v2,"Chemprop v2 (AttentiveFP, ensemble=1)",1.081823,0.940151
8,tdc_vdss_lombardo,conventional,SVR,5.041219,0.135660
9,tdc_ppbr_az,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",11.221405,0.383626


### Cross-dataset model summary

,workflow,model,datasets,mean_rmse,median_rmse,std_rmse,best_rmse,worst_rmse,mean_r2
0,Chemprop v2,"Chemprop v2 (D-MPNN, ensemble=1)",13,7.520273,0.919350,16.153993,0.011649,40.446187,0.643400
1,Chemprop v2,"Chemprop v2 (CMPNN, ensemble=1)",13,7.767333,0.934330,16.638424,0.010590,41.673643,0.625641
2,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",13,9.564432,1.095873,15.009289,0.005171,44.122508,0.531812
3,conventional,MapLight CatBoost,13,10.040744,1.073820,15.363414,0.007872,45.010690,0.405158
4,MapLight + GNN,MapLight + GNN (CatBoost),13,10.445509,1.075697,14.864690,0.008115,43.642589,-0.223538
5,Chemprop v2,"Chemprop v2 (AttentiveFP, ensemble=1)",13,10.675917,0.908513,20.141102,0.006552,40.880091,0.653006
6,conventional,SVR,13,11.014400,1.150743,17.805456,0.033923,49.930858,0.350319
7,conventional,CatBoost,13,11.175845,1.055422,18.466908,0.006671,52.874820,0.454767
8,conventional,Random forest,13,12.423952,1.097767,19.494034,0.009665,54.915557,0.192145
9,conventional,XGBoost,13,13.876325,1.023426,22.298766,0.008023,57.372018,0.100100


In [93]:
# Architecture coverage and prediction-diversity diagnostics


def classify_architecture_family(workflow_name, model_name):
    workflow_text = str(workflow_name or "").strip().lower()
    model_text = str(model_name or "").strip().lower()

    if "ensemble" in workflow_text or model_text.startswith("ensemble ("):
        return "Ensemble meta-model"
    if "chemprop" in workflow_text or "chemprop" in model_text:
        return "Graph neural networks (Chemprop)"
    if "maplight + gnn" in workflow_text or "maplight + gnn" in model_text:
        return "Graph transfer + tabular head"
    if "uni-mol" in workflow_text or "uni-mol" in model_text or "unimol" in model_text:
        return "3D foundation model"
    if "chemml deep learning" in workflow_text or "chemml" in model_text:
        return "Deep tabular neural nets"
    if "tuned conventional" in workflow_text:
        return "Tuned conventional ML"
    if "conventional" in workflow_text:
        return "Conventional ML"
    return "Other / unknown"


def classify_architecture_subfamily(model_name):
    model_text = str(model_name or "").strip().lower()
    if "cmpnn" in model_text:
        return "CMPNN"
    if "attentivefp" in model_text:
        return "AttentiveFP"
    if "d-mpnn" in model_text or "dmpnn" in model_text:
        return "D-MPNN"
    if "maplight + gnn" in model_text:
        return "MapLight + GNN"
    if "chemml mlp" in model_text:
        return "ChemML MLP"
    if "catboost" in model_text:
        return "CatBoost-family"
    if "xgboost" in model_text:
        return "XGBoost-family"
    if "random forest" in model_text:
        return "RandomForest-family"
    if "svr" in model_text:
        return "SVR"
    if "elasticnet" in model_text:
        return "ElasticNet-family"
    if model_text.startswith("ensemble ("):
        return "Stacking / averaging"
    return str(model_name or "")


coverage_source = SUMMARY_METRICS.copy()
error_col = pick_column(coverage_source.columns, ["error", "Error"])
if error_col is not None:
    coverage_source = coverage_source.loc[
        coverage_source[error_col].isna()
        | (coverage_source[error_col].astype(str).str.strip() == "")
    ].copy()

if coverage_source.empty:
    display_note("No successful model rows were available for architecture-coverage diagnostics.")
else:
    coverage_source["architecture_family"] = coverage_source.apply(
        lambda row: classify_architecture_family(row.get("workflow", ""), row.get("model", "")),
        axis=1,
    )
    coverage_source["architecture_subfamily"] = coverage_source["model"].apply(classify_architecture_subfamily)

    family_summary = (
        coverage_source.groupby("architecture_family", dropna=False)
        .agg(
            datasets=("dataset", "nunique"),
            models=("model", "nunique"),
            rows=("model", "size"),
            mean_rmse=(RMSE_COL, "mean") if RMSE_COL in coverage_source.columns else ("model", "size"),
        )
        .reset_index()
        .sort_values(["datasets", "rows"], ascending=[False, False])
    )

    display_note("### Architecture Coverage")
    display(family_summary)

    dataset_family_presence = (
        coverage_source.assign(present=1)
        .groupby(["dataset", "architecture_family"], dropna=False)["present"]
        .max()
        .unstack(fill_value=0)
        .sort_index()
    )

    coverage_fraction = dataset_family_presence.mean(axis=1).rename("coverage_fraction").reset_index()
    coverage_fraction = coverage_fraction.sort_values("coverage_fraction", ascending=False)
    display_note("### Per-dataset architecture-family coverage")
    display(coverage_fraction)

    coverage_fig = px.imshow(
        dataset_family_presence,
        aspect="auto",
        color_continuous_scale=[[0.0, "#f0f0f0"], [1.0, "#2ca25f"]],
        labels={"x": "Architecture family", "y": "Dataset", "color": "Present"},
        title="Architecture-family coverage matrix (dataset x family)",
        text_auto=True,
    )
    coverage_fig.update_layout(height=max(420, 45 * len(dataset_family_presence.index)))
    coverage_fig.show()

    if PREDICTIONS is None or PREDICTIONS.empty:
        display_note("No predictions table was found; skipping architecture-diversity diagnostics.")
    else:
        pred_df = PREDICTIONS.copy()
        dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
        model_col = pick_column(pred_df.columns, ["model", "Model"])
        predicted_col = pick_column(pred_df.columns, ["predicted", "prediction", "y_pred"])
        split_col = pick_column(pred_df.columns, ["split", "Split"])
        smiles_col = pick_column(pred_df.columns, ["smiles", "SMILES", "canonical_smiles"])
        row_col = pick_column(pred_df.columns, ["row_id", "Row", "index"])

        required = [dataset_col, model_col, predicted_col]
        if any(col is None for col in required):
            display_note("Predictions table is missing dataset/model/prediction columns; skipping diversity diagnostics.")
        else:
            pred_df = pred_df.rename(columns={dataset_col: "dataset", model_col: "model", predicted_col: "predicted"})
            if split_col is not None:
                pred_df = pred_df.rename(columns={split_col: "split"})
                pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()

            if smiles_col is not None:
                pred_df = pred_df.rename(columns={smiles_col: "smiles_key"})
            elif row_col is not None:
                pred_df = pred_df.rename(columns={row_col: "smiles_key"})
            else:
                pred_df["smiles_key"] = pred_df.groupby(["dataset", "model"]).cumcount().astype(str)

            pred_df["predicted"] = pd.to_numeric(pred_df["predicted"], errors="coerce")
            pred_df = pred_df.loc[pred_df["predicted"].notna()].copy()

            diversity_rows = []
            pair_rows = []
            for dataset_name, group in pred_df.groupby("dataset", sort=True):
                wide = group.pivot_table(index="smiles_key", columns="model", values="predicted", aggfunc="mean")
                if wide.shape[1] < 2:
                    continue

                corr = wide.corr(method="spearman")
                if corr.shape[0] < 2:
                    continue

                corr_values = corr.to_numpy(dtype=float)
                tri = np.triu_indices_from(corr_values, k=1)
                pair_corr = corr_values[tri]
                pair_corr = pair_corr[np.isfinite(pair_corr)]
                if pair_corr.size == 0:
                    continue

                abs_corr = np.abs(pair_corr)
                diversity_rows.append(
                    {
                        "dataset": dataset_name,
                        "models_in_pairwise_corr": int(corr.shape[0]),
                        "pair_count": int(pair_corr.size),
                        "mean_abs_spearman": float(np.mean(abs_corr)),
                        "median_abs_spearman": float(np.median(abs_corr)),
                        "max_abs_spearman": float(np.max(abs_corr)),
                        "diversity_index": float(1.0 - np.median(abs_corr)),
                    }
                )

                cols = list(corr.columns)
                for i in range(len(cols)):
                    for j in range(i + 1, len(cols)):
                        value = corr.iat[i, j]
                        if pd.isna(value):
                            continue
                        pair_rows.append(
                            {
                                "dataset": dataset_name,
                                "model_a": str(cols[i]),
                                "model_b": str(cols[j]),
                                "spearman": float(value),
                                "abs_spearman": float(abs(value)),
                            }
                        )

            if not diversity_rows:
                display_note("Prediction diversity diagnostics could not be computed (insufficient aligned test predictions).")
            else:
                diversity_df = pd.DataFrame(diversity_rows).sort_values("diversity_index", ascending=False).reset_index(drop=True)
                display_note("### Prediction Diversity Across Architectures")
                display(diversity_df)

                diversity_fig = px.bar(
                    diversity_df.sort_values("diversity_index", ascending=True),
                    x="dataset",
                    y="diversity_index",
                    hover_data=["models_in_pairwise_corr", "median_abs_spearman", "max_abs_spearman"],
                    title="Architecture diversity index by dataset (1 - median |Spearman|)",
                )
                diversity_fig.update_layout(yaxis_title="Diversity index (higher is better)", xaxis_title="Dataset")
                diversity_fig.show()

                if pair_rows:
                    pair_df = pd.DataFrame(pair_rows)
                    pair_df["pair"] = pair_df.apply(
                        lambda row: " / ".join(sorted([str(row["model_a"]), str(row["model_b"])])),
                        axis=1,
                    )
                    pair_summary = (
                        pair_df.groupby("pair", dropna=False)
                        .agg(
                            datasets=("dataset", "nunique"),
                            mean_abs_spearman=("abs_spearman", "mean"),
                            median_abs_spearman=("abs_spearman", "median"),
                            max_abs_spearman=("abs_spearman", "max"),
                        )
                        .reset_index()
                        .sort_values(["mean_abs_spearman", "datasets"], ascending=[False, False])
                    )
                    display_note("### Most redundant model pairs (cross-dataset)")
                    display(pair_summary.head(20))



### Architecture Coverage

,architecture_family,datasets,models,rows,mean_rmse
0,Conventional ML,13,6,75,12.337623
1,Deep tabular neural nets,13,2,26,15.514111
2,Ensemble meta-model,13,1,13,9.564432
4,Graph transfer + tabular head,13,1,13,10.445509
3,Graph neural networks (Chemprop),6,3,16,8.401832


### Per-dataset architecture-family coverage

,dataset,coverage_fraction
0,chemml_cep_homo,1.0
1,chemml_organic_density,1.0
3,freesolv_sampl,1.0
9,tdc_lipophilicity_astrazeneca,1.0
6,tdc_clearance_microsome_az,1.0
11,tdc_solubility_aqsoldb,1.0
2,esol_delaney,0.8
4,tdc_caco2_wang,0.8
5,tdc_clearance_hepatocyte_az,0.8
8,tdc_ld50_zhu,0.8


No predictions table was found; skipping architecture-diversity diagnostics.

In [94]:
# Feature-family representation analysis from selector outputs

feature_family_drop_presence_threshold = 0.20
feature_family_drop_enrichment_threshold = 0.25
feature_family_drop_selected_total_threshold = 25
feature_family_watch_enrichment_threshold = 0.35


def feature_family_from_name(feature_name):
    name = str(feature_name or "")
    if name.startswith("morgan_bit_"):
        return "morgan"
    if name.startswith("ecfp6_bit_"):
        return "ecfp6"
    if name.startswith("fcfp6_bit_"):
        return "fcfp6"
    if name.startswith("layered_bit_"):
        return "layered"
    if name.startswith("atom_pair_bit_"):
        return "atom_pair"
    if name.startswith("topological_torsion_bit_"):
        return "topological_torsion"
    if name.startswith("rdk_path_bit_"):
        return "rdk_path"
    if name.startswith("maccs_bit_"):
        return "maccs"
    if name.startswith("rdkit_"):
        return "rdkit"
    if (
        name.startswith("maplight_morgan_")
        or name.startswith("avalon_count_")
        or name.startswith("erg_")
        or name.startswith("maplight_desc_")
    ):
        return "maplight"
    return "other"


def infer_feature_col(frame):
    return pick_column(frame.columns, ["feature", "Feature", "name"])


def infer_magnitude_col(frame):
    if "abs_coefficient" in frame.columns:
        return "abs_coefficient"
    if "importance" in frame.columns:
        return "importance"
    if "coefficient" in frame.columns:
        return "coefficient"
    return None


selector_rows = []
missing_selector_artifacts = []
for dataset_dir in sorted([p for p in RUN_DIR.iterdir() if p.is_dir()]):
    dataset_name = dataset_dir.name
    selected_path = dataset_dir / "selected_features.csv"
    coeff_path = dataset_dir / "selector_coefficients.csv"
    if not selected_path.exists() or not coeff_path.exists():
        missing_selector_artifacts.append(dataset_name)
        continue

    try:
        selected_df = pd.read_csv(selected_path)
        coeff_df = pd.read_csv(coeff_path)
    except Exception as exc:
        missing_selector_artifacts.append(f"{dataset_name} ({type(exc).__name__})")
        continue

    feature_col = infer_feature_col(coeff_df)
    if feature_col is None:
        missing_selector_artifacts.append(f"{dataset_name} (no feature column)")
        continue

    selected_feature_col = infer_feature_col(selected_df)
    selected_set = set()
    if selected_feature_col is not None:
        selected_set = set(selected_df[selected_feature_col].astype(str).tolist())

    coeff_work = coeff_df.copy()
    coeff_work["feature"] = coeff_work[feature_col].astype(str)
    coeff_work["feature_family"] = coeff_work["feature"].map(feature_family_from_name)
    coeff_work["is_selected"] = coeff_work["feature"].isin(selected_set)

    magnitude_col = infer_magnitude_col(coeff_work)
    if magnitude_col is None:
        coeff_work["feature_magnitude"] = np.nan
    elif magnitude_col == "coefficient":
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce").abs()
    else:
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce")

    grouped = coeff_work.groupby("feature_family", dropna=False)
    for family_name, family_df in grouped:
        selected_mask = family_df["is_selected"].astype(bool)
        selector_rows.append(
            {
                "dataset": dataset_name,
                "feature_family": str(family_name),
                "available_features": int(len(family_df)),
                "selected_features": int(selected_mask.sum()),
                "selected_magnitude_sum": float(
                    family_df.loc[selected_mask, "feature_magnitude"].fillna(0.0).sum()
                ),
            }
        )

if not selector_rows:
    display_note("No selector feature artifacts were found in this run; skipping feature-family representation analysis.")
else:
    selector_family_df = pd.DataFrame(selector_rows)

    family_summary = (
        selector_family_df.groupby("feature_family", dropna=False)
        .agg(
            datasets_with_family=("dataset", "nunique"),
            datasets_selected=("selected_features", lambda s: int((pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum())),
            available_features_total=("available_features", "sum"),
            selected_features_total=("selected_features", "sum"),
            mean_selected_per_dataset=("selected_features", "mean"),
            median_selected_per_dataset=("selected_features", "median"),
            selected_magnitude_total=("selected_magnitude_sum", "sum"),
        )
        .reset_index()
    )

    family_summary["dataset_presence_rate"] = family_summary["datasets_selected"] / family_summary["datasets_with_family"].replace(0, np.nan)
    family_summary["selection_rate"] = family_summary["selected_features_total"] / family_summary["available_features_total"].replace(0, np.nan)

    total_available = float(family_summary["available_features_total"].sum())
    total_selected = float(family_summary["selected_features_total"].sum())
    total_magnitude = float(family_summary["selected_magnitude_total"].sum())

    if total_available > 0 and total_selected > 0:
        family_summary["expected_selected_if_uniform"] = family_summary["available_features_total"] / total_available * total_selected
        family_summary["enrichment_vs_uniform"] = family_summary["selected_features_total"] / family_summary["expected_selected_if_uniform"].replace(0, np.nan)
    else:
        family_summary["expected_selected_if_uniform"] = np.nan
        family_summary["enrichment_vs_uniform"] = np.nan

    family_summary["selected_share"] = family_summary["selected_features_total"] / max(total_selected, 1.0)
    family_summary["selected_magnitude_share"] = family_summary["selected_magnitude_total"] / max(total_magnitude, 1.0)

    family_summary = family_summary.sort_values(["selected_features_total", "selection_rate"], ascending=[False, False]).reset_index(drop=True)

    display_note("### Feature-Family Representation In Selected Features")
    display(family_summary)

    selected_heatmap = (
        selector_family_df.pivot_table(
            index="dataset",
            columns="feature_family",
            values="selected_features",
            aggfunc="sum",
            fill_value=0,
        )
        .sort_index()
    )

    fig = px.imshow(
        selected_heatmap,
        aspect="auto",
        color_continuous_scale="Viridis",
        labels={"x": "Feature family", "y": "Dataset", "color": "# selected"},
        title="Selected feature counts by dataset and family",
        text_auto=True,
    )
    fig.update_layout(height=max(420, 45 * len(selected_heatmap.index)))
    fig.show()

    enrich_plot_df = family_summary.sort_values("enrichment_vs_uniform", ascending=True)
    fig = px.bar(
        enrich_plot_df,
        x="feature_family",
        y="enrichment_vs_uniform",
        hover_data=["selected_features_total", "available_features_total", "dataset_presence_rate", "selection_rate"],
        title="Feature-family selection enrichment vs uniform-by-dimension baseline",
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="black")
    fig.update_layout(xaxis_title="Feature family", yaxis_title="Enrichment (>1 means over-represented)")
    fig.show()

    drop_candidates = family_summary.loc[
        (family_summary["dataset_presence_rate"].fillna(0.0) <= float(feature_family_drop_presence_threshold))
        & (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_drop_enrichment_threshold))
        & (family_summary["selected_features_total"].fillna(0.0) <= float(feature_family_drop_selected_total_threshold))
    ].copy()

    watchlist = family_summary.loc[
        (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_watch_enrichment_threshold))
        & (family_summary["dataset_presence_rate"].fillna(0.0) >= 0.5)
    ].copy()

    if drop_candidates.empty:
        display_note(
            "No feature family met the strict drop-candidate rule in this run. "
            "That means no family looked both rare across datasets and consistently unselected."
        )
    else:
        display_note("### Suggested drop candidates (strict rule)")
        display(
            drop_candidates[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values(["dataset_presence_rate", "selected_features_total"], ascending=[True, True]).reset_index(drop=True)
        )

    if not watchlist.empty:
        display_note("### Watchlist: under-enriched but still used across many datasets")
        display(
            watchlist[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values("enrichment_vs_uniform", ascending=True).reset_index(drop=True)
        )

    if missing_selector_artifacts:
        display_note(
            "Selector artifacts were missing for some datasets and were excluded from this section: "
            + ", ".join(sorted(set(missing_selector_artifacts)))
        )




### Feature-Family Representation In Selected Features

,feature_family,datasets_with_family,datasets_selected,available_features_total,selected_features_total,mean_selected_per_dataset,median_selected_per_dataset,selected_magnitude_total,dataset_presence_rate,selection_rate,expected_selected_if_uniform,enrichment_vs_uniform,selected_share,selected_magnitude_share
0,maplight,13,13,59943,1379,106.076923,22.0,70.559684,1.000000,0.023005,767.585640,1.796542,0.428527,0.191269
1,rdkit,13,10,2821,426,32.769231,8.0,5.748589,0.769231,0.151010,36.123636,11.792833,0.132380,0.015583
2,rdk_path,13,12,26624,293,22.538462,9.0,6.958114,0.923077,0.011005,340.927215,0.859421,0.091050,0.018862
3,atom_pair,13,13,26624,283,21.769231,12.0,57.491189,1.000000,0.010630,340.927215,0.830089,0.087943,0.155843
4,layered,13,10,26624,222,17.076923,6.0,4.108903,0.769231,0.008338,340.927215,0.651165,0.068987,0.011138
5,fcfp6,13,13,26624,213,16.384615,15.0,80.792916,1.000000,0.008000,340.927215,0.624767,0.066190,0.219008
6,ecfp6,13,13,26624,166,12.769231,11.0,79.241123,1.000000,0.006235,340.927215,0.486907,0.051585,0.214802
7,morgan,13,12,26624,90,6.923077,6.0,44.378033,0.923077,0.003380,340.927215,0.263986,0.027968,0.120297
8,maccs,13,8,2171,73,5.615385,2.0,1.884489,0.615385,0.033625,27.800217,2.625879,0.022685,0.005108
9,topological_torsion,13,12,26624,73,5.615385,6.0,17.740393,0.923077,0.002742,340.927215,0.214122,0.022685,0.048090


No feature family met the strict drop-candidate rule in this run. That means no family looked both rare across datasets and consistently unselected.

### Watchlist: under-enriched but still used across many datasets

,feature_family,datasets_selected,datasets_with_family,dataset_presence_rate,selected_features_total,available_features_total,selection_rate,enrichment_vs_uniform
0,topological_torsion,12,13,0.923077,73,26624,0.002742,0.214122
1,morgan,12,13,0.923077,90,26624,0.003380,0.263986


Selector artifacts were missing for some datasets and were excluded from this section: lipophilicity

In [95]:
# Leaderboard-relative performance summary (using cached run artifacts)

def leaderboard_metric_kind(text):
    t = str(text).strip().lower()
    if not t:
        return None
    if "mean_squared_error" in t or t == "mse" or " mse" in t:
        return "mse"
    if "rmse" in t:
        return "rmse"
    if "mae" in t:
        return "mae"
    if "spearman" in t:
        return "spearman"
    if "pearson" in t:
        return "pearson"
    if t in {"r2", "r^2", "r?"} or "r2" in t or "r^2" in t or "r?" in t:
        return "r2"
    return t


def leaderboard_direction(metric_kind):
    if metric_kind in {"mse", "rmse", "mae"}:
        return "lower"
    if metric_kind in {"r2", "spearman", "pearson"}:
        return "higher"
    return "lower"


def model_metric_column(frame, metric_kind):
    if metric_kind in {"mse", "rmse"}:
        return metric_column(frame, "AUTO", metric_hint="rmse")
    if metric_kind == "r2":
        return metric_column(frame, "AUTO", metric_hint="r2")
    if metric_kind == "mae":
        return pick_column(frame.columns, ["test_mae", "Test MAE", "cv_mae", "CV MAE", "mae", "MAE"])
    if metric_kind == "spearman":
        return pick_column(frame.columns, ["test_spearman", "Test Spearman", "cv_spearman", "CV Spearman", "spearman", "Spearman"])
    if metric_kind == "pearson":
        return pick_column(frame.columns, ["test_pearson", "Test Pearson", "cv_pearson", "CV Pearson", "pearson", "Pearson"])
    return None


def parse_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def to_leaderboard_scale(value, metric_kind):
    value = float(value)
    if metric_kind == "mse":
        return value ** 2
    return value


leaderboard_eval = None
if LEADERBOARD_COMPARISON is not None and not LEADERBOARD_COMPARISON.empty:
    leaderboard_eval = LEADERBOARD_COMPARISON.copy()
    if "leaderboard_rank" in leaderboard_eval.columns and "estimated_rank_vs_top10" not in leaderboard_eval.columns:
        leaderboard_eval = leaderboard_eval.rename(columns={"leaderboard_rank": "estimated_rank_vs_top10"})
else:
    ref = LEADERBOARD_TOP10.copy() if LEADERBOARD_TOP10 is not None else None
    if ref is not None and not ref.empty and "is_top10_entry" in ref.columns:
        ref = ref.loc[ref["is_top10_entry"].fillna(False)].copy()

    rows = []
    if ref is not None and not ref.empty:
        for dataset_name, dataset_frame in SUMMARY_METRICS.groupby("dataset", sort=True):
            dataset_frame = dataset_frame.copy()

            benchmark_id = ""
            if "benchmark_id" in dataset_frame.columns:
                non_empty_ids = dataset_frame["benchmark_id"].dropna().astype(str)
                non_empty_ids = non_empty_ids[non_empty_ids.str.strip() != ""]
                if not non_empty_ids.empty:
                    benchmark_id = non_empty_ids.iloc[0]

            ref_subset = pd.DataFrame()
            if benchmark_id:
                ref_subset = ref.loc[ref["benchmark_id"].astype(str) == str(benchmark_id)].copy() if "benchmark_id" in ref.columns else pd.DataFrame()
            if ref_subset.empty and "dataset" in ref.columns:
                ref_subset = ref.loc[ref["dataset"].astype(str) == str(dataset_name)].copy()
            if ref_subset.empty:
                continue

            metric_name = str(ref_subset.get("leaderboard_metric_name", pd.Series([""])).dropna().iloc[0]) if "leaderboard_metric_name" in ref_subset.columns else ""
            metric_kind = leaderboard_metric_kind(metric_name)
            metric_col = model_metric_column(dataset_frame, metric_kind)
            if metric_col is None or metric_col not in dataset_frame.columns:
                continue

            candidate = dataset_frame.copy()
            if "error" in candidate.columns:
                candidate = candidate.loc[candidate["error"].isna() | (candidate["error"].astype(str).str.strip() == "")].copy()
            candidate[metric_col] = parse_numeric(candidate[metric_col])
            candidate = candidate.loc[candidate[metric_col].notna()].copy()
            if candidate.empty:
                continue
            candidate["_leaderboard_scale_value"] = candidate[metric_col].apply(lambda v: to_leaderboard_scale(v, metric_kind))

            direction = leaderboard_direction(metric_kind)
            if direction == "lower":
                best_idx = candidate["_leaderboard_scale_value"].idxmin()
            else:
                best_idx = candidate["_leaderboard_scale_value"].idxmax()
            best_row = candidate.loc[best_idx]
            best_value_raw = float(best_row[metric_col])
            best_value = float(best_row["_leaderboard_scale_value"])

            metric_values = parse_numeric(ref_subset.get("metric_value_numeric", pd.Series(dtype=float))).dropna().to_numpy(dtype=float)
            if len(metric_values) == 0:
                continue

            if direction == "lower":
                top1 = float(np.min(metric_values))
                cutoff = float(np.max(metric_values))
                estimated_rank = 1 + int(np.sum(metric_values < best_value))
                gap_top1 = best_value - top1
                gap_top10 = best_value - cutoff
            else:
                top1 = float(np.max(metric_values))
                cutoff = float(np.min(metric_values))
                estimated_rank = 1 + int(np.sum(metric_values > best_value))
                gap_top1 = top1 - best_value
                gap_top10 = cutoff - best_value

            rows.append(
                {
                    "dataset": dataset_name,
                    "benchmark_id": benchmark_id,
                    "leaderboard_metric_name": metric_name,
                    "metric_kind": metric_kind,
                    "direction": direction,
                    "best_model": str(best_row.get("model", "")),
                    "best_workflow": str(best_row.get("workflow", "")),
                    "best_value": best_value,
                    "best_value_raw_model_metric": best_value_raw,
                    "best_value_metric_column": metric_col,
                    "leaderboard_top1": top1,
                    "leaderboard_top10_cutoff": cutoff,
                    "estimated_rank_vs_top10": int(estimated_rank) if estimated_rank <= 10 else 11,
                    "in_top10_estimate": bool(estimated_rank <= 10),
                    "gap_to_top1": float(gap_top1),
                    "gap_to_top10_cutoff": float(gap_top10),
                }
            )

    leaderboard_eval = pd.DataFrame(rows)

if leaderboard_eval is None or leaderboard_eval.empty:
    display_note("No leaderboard comparison was available for this run.")
else:
    leaderboard_eval = leaderboard_eval.copy()
    if "estimated_rank_vs_top10" in leaderboard_eval.columns:
        leaderboard_eval["estimated_rank_vs_top10"] = pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce")
        leaderboard_eval["estimated_rank_label"] = leaderboard_eval["estimated_rank_vs_top10"].apply(lambda x: ">10" if pd.notna(x) and int(x) > 10 else (str(int(x)) if pd.notna(x) else "NA"))
        if "in_top10_estimate" not in leaderboard_eval.columns:
            leaderboard_eval["in_top10_estimate"] = leaderboard_eval["estimated_rank_vs_top10"].fillna(11).astype(float) <= 10

    total = int(len(leaderboard_eval))
    in_top10 = int(pd.to_numeric(leaderboard_eval.get("in_top10_estimate", pd.Series(dtype=float)), errors="coerce").fillna(0).astype(int).sum())
    top1_count = int((pd.to_numeric(leaderboard_eval.get("estimated_rank_vs_top10", pd.Series(dtype=float)), errors="coerce") == 1).sum())

    display_note(
        f"### Leaderboard Performance Overview\n"
        f"Compared datasets: **{total}**  \n"
        f"Estimated in top-10: **{in_top10}/{total}**  \n"
        f"Estimated #1: **{top1_count}**"
    )

    display_cols = [
        "dataset",
        "leaderboard_metric_name",
        "best_model",
        "best_workflow",
        "best_value",
        "best_value_raw_model_metric",
        "best_value_metric_column",
        "leaderboard_top1",
        "leaderboard_top10_cutoff",
        "estimated_rank_label",
        "in_top10_estimate",
        "gap_to_top1",
        "gap_to_top10_cutoff",
    ]
    display_cols = [c for c in display_cols if c in leaderboard_eval.columns]
    display(
        leaderboard_eval[display_cols]
        .sort_values(by=["in_top10_estimate", "gap_to_top10_cutoff"], ascending=[False, True])
        .reset_index(drop=True)
    )

    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        plot_df = leaderboard_eval.sort_values("gap_to_top10_cutoff", ascending=True).copy()
        fig = px.bar(
            plot_df,
            x="dataset",
            y="gap_to_top10_cutoff",
            color="metric_kind" if "metric_kind" in plot_df.columns else None,
            hover_data=[col for col in ["best_model", "best_value", "leaderboard_top10_cutoff", "estimated_rank_label"] if col in plot_df.columns],
            title="Gap to leaderboard top-10 cutoff (negative = better than cutoff)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(yaxis_title="Gap to top-10 cutoff", xaxis_title="Dataset")
        fig.show()

    if "estimated_rank_label" in leaderboard_eval.columns:
        rank_order = [str(i) for i in range(1, 11)] + [">10"]
        rank_counts = (
            leaderboard_eval.groupby("estimated_rank_label", dropna=False)
            .size()
            .reset_index(name="datasets")
        )
        rank_counts["estimated_rank_label"] = pd.Categorical(rank_counts["estimated_rank_label"], categories=rank_order, ordered=True)
        rank_counts = rank_counts.sort_values("estimated_rank_label")
        fig = px.bar(rank_counts, x="estimated_rank_label", y="datasets", title="Estimated leaderboard rank distribution")
        fig.update_layout(xaxis_title="Estimated rank vs top-10", yaxis_title="Dataset count")
        fig.show()


### Leaderboard Performance Overview
Compared datasets: **10**  
Estimated in top-10: **9/10**  
Estimated #1: **1**

,dataset,leaderboard_metric_name,best_model,best_workflow,best_value,best_value_raw_model_metric,best_value_metric_column,leaderboard_top1,leaderboard_top10_cutoff,estimated_rank_label,in_top10_estimate,gap_to_top1,gap_to_top10_cutoff
0,tdc_ppbr_az,MAE,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,7.529225,7.529225,test_mae,7.4400,8.6800,3,True,0.089225,-1.150775
1,esol_delaney,Test RMSE,MapLight CatBoost,conventional,0.618938,0.618938,test_rmse,0.8851,1.7406,1,True,-0.266162,-1.121662
2,tdc_clearance_hepatocyte_az,Spearman,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.534741,0.534741,test_spearman,0.5360,0.4300,2,True,0.001259,-0.104741
3,tdc_solubility_aqsoldb,MAE,XGBoost,conventional,0.747317,0.747317,test_mae,0.7410,0.8290,2,True,0.006317,-0.081683
4,tdc_lipophilicity_astrazeneca,MAE,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.478964,0.478964,test_mae,0.4560,0.5390,4,True,0.022964,-0.060036
5,tdc_ld50_zhu,MAE,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.588839,0.588839,test_mae,0.5520,0.6300,5,True,0.036839,-0.041161
6,tdc_caco2_wang,MAE,MapLight CatBoost,conventional,0.294583,0.294583,test_mae,0.2560,0.3260,8,True,0.038583,-0.031417
7,tdc_vdss_lombardo,Spearman,MapLight + GNN (CatBoost),MapLight + GNN,0.519044,0.519044,test_spearman,0.7130,0.4970,10,True,0.193956,-0.022044
8,tdc_clearance_microsome_az,Spearman,MapLight + GNN (CatBoost),MapLight + GNN,0.586422,0.586422,test_spearman,0.6300,0.5780,8,True,0.043578,-0.008422
9,tdc_half_life_obach,Spearman,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.308258,0.308258,test_spearman,0.5760,0.3720,>10,False,0.267742,0.063742


In [96]:
# Run completeness and leaderboard-reference context

if RUN_STATUS is None or RUN_STATUS.empty:
    display_note("No per-dataset `run_status.json` files were found in this run directory.")
else:
    status_df = RUN_STATUS.copy()
    status_df["status"] = status_df["status"].astype(str)
    display_note("### Run completeness by dataset")
    display(status_df.sort_values(["status", "dataset"], ascending=[True, True]).reset_index(drop=True))

    status_counts = status_df.groupby("status", dropna=False).size().reset_index(name="datasets")
    fig = px.bar(status_counts, x="status", y="datasets", title="Run status counts")
    fig.update_layout(xaxis_title="Status", yaxis_title="Dataset count")
    fig.show()

if LEADERBOARD_TOP10 is None or LEADERBOARD_TOP10.empty:
    display_note("No `leaderboard_top10_reference.csv` was found for this run.")
else:
    ref = LEADERBOARD_TOP10.copy()
    if "is_top10_entry" in ref.columns:
        ref = ref.loc[ref["is_top10_entry"].fillna(False)].copy()

    display_note("### Leaderboard reference coverage")
    coverage_cols = [
        "dataset",
        "benchmark_id",
        "benchmark_suite",
        "leaderboard_metric_name",
        "leaderboard_dataset_split",
        "rank",
        "model",
        "metric_value",
    ]
    coverage_cols = [c for c in coverage_cols if c in ref.columns]
    display(ref[coverage_cols].head(30))

    coverage_summary = (
        ref.groupby([c for c in ["dataset", "leaderboard_metric_name"] if c in ref.columns], dropna=False)
        .agg(
            top10_rows=("model", "size"),
            top10_best=("metric_value_numeric", "min"),
            top10_worst=("metric_value_numeric", "max"),
        )
        .reset_index()
    )
    display(coverage_summary.sort_values("dataset").reset_index(drop=True))


### Run completeness by dataset

,dataset,status,checkpoint_stage,n_metrics_rows,started_at,completed_at,elapsed_seconds
0,chemml_cep_homo,completed,,13.0,,2026-04-19 23:10:53,1240.134
1,chemml_organic_density,completed,,13.0,,2026-04-19 22:50:13,893.087
2,esol_delaney,completed,,14.0,,2026-04-21 09:30:58,1916.425
3,freesolv_sampl,completed,,13.0,,2026-04-21 09:42:32,693.861
4,tdc_caco2_wang,completed,,14.0,,2026-04-20 01:05:52,6899.136
5,tdc_clearance_hepatocyte_az,completed,,14.0,,2026-04-21 02:59:00,9471.740
6,tdc_clearance_microsome_az,completed,,13.0,,2026-04-21 04:04:53,3952.726
7,tdc_half_life_obach,completed,,14.0,,2026-04-21 00:21:08,3404.584
8,tdc_ld50_zhu,completed,,14.0,,2026-04-21 08:59:01,17648.443
9,tdc_lipophilicity_astrazeneca,completed,,13.0,,2026-04-20 05:27:06,15674.035


### Leaderboard reference coverage

,dataset,benchmark_id,benchmark_suite,leaderboard_metric_name,leaderboard_dataset_split,rank,model,metric_value
0,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,1,CaliciBoost,0.256 ± 0.006
1,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,2,XG Boost,0.274 ± 0.004
2,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,3,MapLight,0.276 ± 0.005
3,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,4,BaseBoosting,0.285 ± 0.005
4,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,5,MolMapNet-D,0.287 ± 0.005
5,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,6,MapLight + GNN,0.287 ± 0.005
6,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,7,XGBoost,0.289 ± 0.011
7,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,8,DeepMol (AutoML),0.297 ± 0.008
8,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,9,Basic ML,0.321 ± 0.005
9,tdc_caco2_wang,caco2_wang,tdc,MAE,Scaffold,10,ADMETrix,0.326 ± 0.042


,dataset,leaderboard_metric_name,top10_rows,top10_best,top10_worst
0,esol_delaney,Test RMSE,2,0.8851,1.7406
1,lipophilicity,Test RMSE,2,0.7806,0.9621
2,polaris_adme_fang_hppb_1,mean_squared_error,10,0.1430,0.3830
3,polaris_adme_fang_perm_1,mean_squared_error,10,0.1130,0.2570
4,polaris_adme_fang_rclint_1,mean_squared_error,10,0.2160,0.4030
5,polaris_adme_fang_rppb_1,mean_squared_error,10,0.2300,0.6340
6,polaris_adme_fang_solu_1,mean_squared_error,10,0.2220,0.3290
7,tdc_caco2_wang,MAE,10,0.2560,0.3260
8,tdc_clearance_hepatocyte_az,Spearman,10,0.4300,0.5360
9,tdc_clearance_microsome_az,Spearman,10,0.5780,0.6300


In [97]:
heatmap_source = TEST_RMSE_PIVOT.copy() if TEST_RMSE_PIVOT is not None else None
if heatmap_source is None or heatmap_source.empty:
    heatmap_source = SUMMARY_METRICS.pivot_table(index="dataset", columns="model", values=RMSE_COL, aggfunc="mean").reset_index()

if "dataset" not in heatmap_source.columns:
    first_col = heatmap_source.columns[0]
    heatmap_source = heatmap_source.rename(columns={first_col: "dataset"})

heatmap_df = heatmap_source.set_index("dataset")
fig = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="Viridis_r",
    labels={"x": "Model", "y": "Dataset", "color": RMSE_COL},
    text_auto='.3f',
    title="Held-out RMSE by dataset and model",
)
fig.update_layout(height=max(450, 60 + 40 * len(heatmap_df.index)))
fig.show()

box_df = SUMMARY_METRICS.copy()
fig = px.box(
    box_df,
    x="model",
    y=RMSE_COL,
    color="workflow" if "workflow" in box_df.columns else None,
    points="all",
    title="RMSE distribution across datasets",
)
fig.update_layout(xaxis_title="Model", yaxis_title=RMSE_COL)
fig.show()

bar_df = model_summary.sort_values("mean_rmse", ascending=True).head(int(show_top_n_models)).copy()
fig = px.bar(
    bar_df,
    x="model",
    y="mean_rmse",
    color="workflow" if "workflow" in bar_df.columns else None,
    error_y="std_rmse" if "std_rmse" in bar_df.columns else None,
    title="Mean held-out RMSE by model",
)
fig.update_layout(xaxis_title="Model", yaxis_title="Mean RMSE")
fig.show()


In [98]:
if PREDICTIONS is not None and not PREDICTIONS.empty:
    pred_df = PREDICTIONS.copy()
    dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(pred_df.columns, ["model", "Model"])
    observed_col = pick_column(pred_df.columns, ["observed", "y_true", "actual"])
    predicted_col = pick_column(pred_df.columns, ["predicted", "y_pred", "prediction"])
    split_col = pick_column(pred_df.columns, ["split", "Split"])

    if all(col is not None for col in [dataset_col, model_col, observed_col, predicted_col]):
        pred_df = pred_df.rename(columns={
            dataset_col: "dataset",
            model_col: "model",
            observed_col: "observed",
            predicted_col: "predicted",
        })
        if split_col is not None:
            pred_df = pred_df.rename(columns={split_col: "split"})
            pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()
        if len(pred_df) > int(prediction_sample_cap):
            pred_df = pred_df.sample(int(prediction_sample_cap), random_state=42)

        fig = px.scatter(
            pred_df,
            x="observed",
            y="predicted",
            color="model",
            facet_col="dataset",
            facet_col_wrap=3,
            opacity=0.65,
            title="Observed vs predicted on held-out rows",
        )
        low = float(np.nanmin([pred_df["observed"].min(), pred_df["predicted"].min()]))
        high = float(np.nanmax([pred_df["observed"].max(), pred_df["predicted"].max()]))
        fig.add_trace(go.Scatter(x=[low, high], y=[low, high], mode="lines", name="Ideal", line=dict(color="black", dash="dash")))
        fig.show()
    else:
        display_note("Predictions table exists, but the expected observed/predicted columns were not found.")
else:
    display_note("No aggregate predictions table was found for this run.")

if GA_HISTORY is not None and not GA_HISTORY.empty:
    ga_df = GA_HISTORY.copy()
    dataset_col = pick_column(ga_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(ga_df.columns, ["model", "Model"])
    generation_col = pick_column(ga_df.columns, ["generation", "Generation"])
    best_col = pick_column(ga_df.columns, ["best_fitness", "best_rmse", "best_score", "Best fitness"])
    if all(col is not None for col in [dataset_col, model_col, generation_col, best_col]):
        ga_df = ga_df.rename(columns={dataset_col: "dataset", model_col: "model", generation_col: "generation", best_col: "best_value"})
        fig = px.line(ga_df, x="generation", y="best_value", color="model", facet_col="dataset", facet_col_wrap=3, title="GA convergence history")
        fig.update_layout(yaxis_title=best_col)
        fig.show()
    else:
        display_note("GA history exists, but the expected columns were not found for plotting.")


No aggregate predictions table was found for this run.